# Merchant Trust Platform

AI-assisted commerce safety system for merchant onboarding and fraud detection.

## 1. Load and Explore Data

In [2]:
import pandas as pd

df = pd.read_csv("../data/merchants.csv")
df.head()

,merchant_id,merchant_name,country,category,days_since_signup,verification_status,avg_order_value,num_orders,chargeback_rate,refund_rate,traffic_source,is_high_risk_label
0,M001,UrbanWear Co,US,Fashion,120,verified,45.2,3200,0.02,0.05,organic,0
1,M002,QuickElectro Deals,CN,Electronics,10,unverified,120.5,200,0.15,0.20,ads,1
2,M003,FreshHome Goods,DE,Home,300,verified,60.0,5400,0.01,0.03,organic,0
3,M004,FlashTech Hub,NG,Electronics,5,unverified,250.0,80,0.30,0.25,social,1
4,M005,StyleCorner,FR,Fashion,90,verified,35.0,2100,0.03,0.04,organic,0


## 2. Rule-Based Risk Scoring

In [3]:
def compute_risk(row):
    risk = 0
    
    # Verification risk
    if row["verification_status"] == "unverified":
        risk += 25
    
    # Chargeback risk
    if row["chargeback_rate"] > 0.1:
        risk += 35
    
    # Refund abuse
    if row["refund_rate"] > 0.15:
        risk += 20
    
    # New merchant risk
    if row["days_since_signup"] < 30:
        risk += 20
    
    # High-value anomaly
    if row["avg_order_value"] > 150:
        risk += 10
    
    return min(risk, 100)

In [4]:
df["risk_score"] = df.apply(compute_risk, axis=1)
df[["merchant_name", "risk_score", "is_high_risk_label"]]

,merchant_name,risk_score,is_high_risk_label
0,UrbanWear Co,0,0
1,QuickElectro Deals,100,1
2,FreshHome Goods,0,0
3,FlashTech Hub,100,1
4,StyleCorner,0,0
5,BargainKing Store,100,1
6,GreenGarden Supplies,0,0
7,DigitalPro Market,100,1
8,Everyday Essentials,0,0
9,Trendify Shop,100,1


## 3. Risk Tier Classification

In [5]:
def risk_level(score):
    if score < 30:
        return "LOW"
    elif score < 70:
        return "MEDIUM"
    else:
        return "HIGH"

df["risk_level"] = df["risk_score"].apply(risk_level)

df[["merchant_name", "risk_score", "risk_level"]]

,merchant_name,risk_score,risk_level
0,UrbanWear Co,0,LOW
1,QuickElectro Deals,100,HIGH
2,FreshHome Goods,0,LOW
3,FlashTech Hub,100,HIGH
4,StyleCorner,0,LOW
5,BargainKing Store,100,HIGH
6,GreenGarden Supplies,0,LOW
7,DigitalPro Market,100,HIGH
8,Everyday Essentials,0,LOW
9,Trendify Shop,100,HIGH


In [6]:
pd.crosstab(df["risk_level"], df["is_high_risk_label"])

is_high_risk_label,0,1
risk_level,,
HIGH,0,5
LOW,5,0


## 4. Machine Learning Model

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Convert categorical variables into numeric
df_ml = df.copy()

df_ml["verification_status"] = df_ml["verification_status"].map({
    "verified": 0,
    "unverified": 1
})

df_ml["traffic_source"] = df_ml["traffic_source"].map({
    "organic": 0,
    "ads": 1,
    "social": 2
})

df_ml["category"] = df_ml["category"].map({
    "Fashion": 0,
    "Electronics": 1,
    "Home": 2,
    "General": 3
})

In [9]:
features = [
    "days_since_signup",
    "verification_status",
    "avg_order_value",
    "num_orders",
    "chargeback_rate",
    "refund_rate",
    "traffic_source",
    "category"
]

X = df_ml[features]
y = df_ml["is_high_risk_label"]

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [11]:
model = LogisticRegression()
model.fit(X_train, y_train)

LogisticRegression()

In [12]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         2

    accuracy                           1.00         3
   macro avg       1.00      1.00      1.00         3
weighted avg       1.00      1.00      1.00         3



## 5. Hybrid Risk Scoring

In [13]:
import numpy as np

# ML probabilities
ml_probs = model.predict_proba(X)[:, 1]

df_ml["ml_risk_score"] = ml_probs * 100
df_ml["rule_risk_score"] = df["risk_score"]

# Hybrid score (weighted system)
df_ml["hybrid_risk_score"] = (
    0.6 * df_ml["ml_risk_score"] +
    0.4 * df_ml["rule_risk_score"]
)

df_ml[["merchant_name", "ml_risk_score", "rule_risk_score", "hybrid_risk_score"]]

,merchant_name,ml_risk_score,rule_risk_score,hybrid_risk_score
0,UrbanWear Co,3.879098e-10,0,2.327459e-10
1,QuickElectro Deals,9.999982e+01,100,9.999989e+01
2,FreshHome Goods,1.043560e-22,0,6.261361e-23
3,FlashTech Hub,9.999997e+01,100,9.999998e+01
4,StyleCorner,7.248650e-04,0,4.349190e-04
5,BargainKing Store,9.999990e+01,100,9.999994e+01
6,GreenGarden Supplies,1.107400e-38,0,6.644400e-39
7,DigitalPro Market,9.999938e+01,100,9.999963e+01
8,Everyday Essentials,6.289908e-49,0,3.773945e-49
9,Trendify Shop,9.999986e+01,100,9.999992e+01


In [14]:
def hybrid_risk_level(score):
    if score < 30:
        return "LOW"
    elif score < 70:
        return "MEDIUM"
    else:
        return "HIGH"

df_ml["final_risk_level"] = df_ml["hybrid_risk_score"].apply(hybrid_risk_level)

df_ml[["merchant_name", "hybrid_risk_score", "final_risk_level"]]

,merchant_name,hybrid_risk_score,final_risk_level
0,UrbanWear Co,2.327459e-10,LOW
1,QuickElectro Deals,9.999989e+01,HIGH
2,FreshHome Goods,6.261361e-23,LOW
3,FlashTech Hub,9.999998e+01,HIGH
4,StyleCorner,4.349190e-04,LOW
5,BargainKing Store,9.999994e+01,HIGH
6,GreenGarden Supplies,6.644400e-39,LOW
7,DigitalPro Market,9.999963e+01,HIGH
8,Everyday Essentials,3.773945e-49,LOW
9,Trendify Shop,9.999992e+01,HIGH


## 6. Merchant Onboarding Decisions

In [15]:
def onboarding_decision(score):
    if score < 30:
        return "APPROVE"
    elif score < 70:
        return "SEND_TO_REVIEW"
    else:
        return "REJECT"

df_ml["decision"] = df_ml["hybrid_risk_score"].apply(onboarding_decision)

df_ml[[
    "merchant_name",
    "hybrid_risk_score",
    "final_risk_level",
    "decision"
]]

,merchant_name,hybrid_risk_score,final_risk_level,decision
0,UrbanWear Co,2.327459e-10,LOW,APPROVE
1,QuickElectro Deals,9.999989e+01,HIGH,REJECT
2,FreshHome Goods,6.261361e-23,LOW,APPROVE
3,FlashTech Hub,9.999998e+01,HIGH,REJECT
4,StyleCorner,4.349190e-04,LOW,APPROVE
5,BargainKing Store,9.999994e+01,HIGH,REJECT
6,GreenGarden Supplies,6.644400e-39,LOW,APPROVE
7,DigitalPro Market,9.999963e+01,HIGH,REJECT
8,Everyday Essentials,3.773945e-49,LOW,APPROVE
9,Trendify Shop,9.999992e+01,HIGH,REJECT


In [16]:
df_ml["decision"].value_counts()

decision
APPROVE    5
REJECT     5
Name: count, dtype: int64

In [17]:
pd.crosstab(df_ml["decision"], df_ml["is_high_risk_label"])

is_high_risk_label,0,1
decision,,
APPROVE,5,0
REJECT,0,5


## 7. Performance Analysis

In [18]:
approval_rate = (
    (df_ml["decision"] == "APPROVE").mean() * 100
)

print(f"Approval Rate: {approval_rate:.1f}%")

Approval Rate: 50.0%


In [19]:
review_rate = (
    (df_ml["decision"] == "SEND_TO_REVIEW").mean() * 100
)

print(f"Review Rate: {review_rate:.1f}%")

Review Rate: 0.0%


In [20]:
rejection_rate = (
    (df_ml["decision"] == "REJECT").mean() * 100
)

print(f"Rejection Rate: {rejection_rate:.1f}%")

Rejection Rate: 50.0%


## 8. Risk Explanations

In [22]:
def explain_risk(row):
    reasons = []

    if row["verification_status"] == "unverified":
        reasons.append("Unverified merchant")

    if row["chargeback_rate"] > 0.1:
        reasons.append("High chargeback rate")

    if row["refund_rate"] > 0.15:
        reasons.append("High refund rate")

    if row["days_since_signup"] < 30:
        reasons.append("New merchant")

    if row["avg_order_value"] > 150:
        reasons.append("High average order value")

    return ", ".join(reasons)

In [23]:
df_ml["risk_reasons"] = df.apply(explain_risk, axis=1)

df_ml[
    [
        "merchant_name",
        "hybrid_risk_score",
        "decision",
        "risk_reasons"
    ]
]

,merchant_name,hybrid_risk_score,decision,risk_reasons
0,UrbanWear Co,2.327459e-10,APPROVE,
1,QuickElectro Deals,9.999989e+01,REJECT,"Unverified merchant, High chargeback rate, Hig..."
2,FreshHome Goods,6.261361e-23,APPROVE,
3,FlashTech Hub,9.999998e+01,REJECT,"Unverified merchant, High chargeback rate, Hig..."
4,StyleCorner,4.349190e-04,APPROVE,
5,BargainKing Store,9.999994e+01,REJECT,"Unverified merchant, High chargeback rate, Hig..."
6,GreenGarden Supplies,6.644400e-39,APPROVE,
7,DigitalPro Market,9.999963e+01,REJECT,"Unverified merchant, High chargeback rate, Hig..."
8,Everyday Essentials,3.773945e-49,APPROVE,
9,Trendify Shop,9.999992e+01,REJECT,"Unverified merchant, High chargeback rate, Hig..."
